# SciPulse Insights — Spark, score et contrôles du calcul

Périmètre : traitement Spark, Gold, orchestration Airflow et dashboard Kibana augmenté inclus. Les mesures et plans proviennent du job exécuté sur les trois sources Silver MinIO.

In [ ]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'reports').exists(): ROOT = ROOT.parent


In [ ]:
results = json.loads((ROOT / 'reports/spark_results.json').read_text(encoding='utf-8'))
{k:v for k,v in results.items() if k != 'top_articles'}

## Couverture observée

4958 articles OpenAlex, 81 articles HN, 13 communs aux trois sources.

In [ ]:
plans = json.loads((ROOT / 'reports/spark_plans.json').read_text(encoding='utf-8'))
print(plans['broadcast'])
print(plans['sort_merge'])

## Choix des jointures

Jointures gauches pour conserver les articles sans signal. Dimensions dédupliquées et HN agrégé avant broadcast. La comparaison porte sur les plans, sans prétendre à un benchmark chronométré.

## Score

`100 × (0.50 × citations_log_normalisées + 0.30 × score_HN_log_normalisé + 0.20 × exp(-âge/5))`. Les citations inconnues restent nulles ; `score_is_lower_bound` identifie la borne basse. Les poids sont justifiés dans la méthodologie.

In [ ]:
gold = pd.read_parquet(ROOT / 'reports/gold_summary.parquet')
assert gold.arxiv_id.is_unique
assert gold.impact_score.between(0,100).all()
assert gold.loc[~gold.citation_matched, 'citation_count'].isna().all()
gold.sort_values('impact_score',ascending=False).head(10)

![Couverture](../artifacts/figures/coverage.png)

![Score](../artifacts/figures/impact.png)

In [ ]:
cluster = json.loads((ROOT / 'reports/spark_cluster_execution.json').read_text(encoding='utf-8'))
assert cluster['two_workers_used']
[(e['hostPort'], e['completedTasks'], e['failedTasks']) for e in cluster['executors']]

## Validation hors ajustement

Le job transforme trois abstracts nouveaux, dont un vide et un terme inconnu. Il vérifie des vecteurs TF-IDF finis de 2 048 dimensions. Les tests Python couvrent aussi les formats d’identifiants et les valeurs manquantes. Il ne s’agit pas d’une évaluation de précision prédictive.

## Conclusions

La couverture doit être affichée avec le score. Prolonger la collecte HN et enrichir davantage de DOI avant de conclure sur les différences entre communautés.

## Gold et intégration Airflow

Le DataFrame enrichi est écrit avec Elasticsearch-Hadoop. Airflow soumet Spark via un BashOperator puis contrôle la Gold avant publication Kibana. Les rapports ci-dessous décrivent l'exécution enregistrée.

In [ ]:
validation = json.loads((ROOT / 'reports/gold_validation.json').read_text(encoding='utf-8'))
validation['checks']

In [ ]:
run = json.loads((ROOT / 'reports/airflow_cluster_execution.json').read_text(encoding='utf-8'))
[(t['task_id'], t['state']) for t in run['task_instances']]

## Dashboard Kibana augmenté

Le dashboard **SciPulse Insights** lit uniquement l'index Gold `arxiv-papers-enriched`. Il présente le nombre d'articles, la couverture OpenAlex et HN, le score d'impact moyen, les catégories, les publications par année, les profils d'impact, la croissance des publications et un tableau d'articles trié par score.

Adresse : http://localhost:5601/app/dashboards#/view/scipulse-insights

Dans la barre KQL, appliquer successivement :

```text
citation_count >= 50 and hn_matched: false
```

Ce filtre sélectionne les articles très cités sans mention HN observée dans la collecte.

```text
hn_score >= 50 and citation_count < 10
```

Ce filtre sélectionne les articles populaires sur HN et peu cités parmi ceux dont les citations sont connues. Les seuils 50 et 10 sont des choix de démonstration.

Les deux filtres sélectionnent respectivement 881 et 3 articles dans les captures. Le rapport navigateur enregistré dans `reports/browser_checks.json` ne relevait aucune erreur visible sur le dashboard et confirmait 4 063 articles pour le filtre `primary_category: "cs.AI"`. Ce rapport décrit une vérification antérieure, pas une surveillance continue.

**Limite :** le tableau de croissance utilise `growth_pct`, calculé sur les publications. Il ne classe pas les sous-domaines selon une croissance combinée des citations et du signal HN. Le panneau ne mesure donc pas cette croissance combinée.

## Captures du dashboard

Les six captures `scipulse-kibana0.png` à `scipulse-kibana5.png` illustrent les panneaux et les résultats filtrés. Voir [leur inventaire](../artifacts/screenshots/README.md). Les résultats filtrés visibles sont de 881 articles très cités sans HN observé et 3 articles populaires HN peu cités.

![Vue des compteurs](../artifacts/screenshots/scipulse-kibana0.png)

![Articles très cités sans HN observé : 881](../artifacts/screenshots/scipulse-kibana4.png)

![Articles populaires HN peu cités : 3](../artifacts/screenshots/scipulse-kibana5.png)


## Monitoring du pipeline

Le dashboard [Monitoring Bronze → Silver → Gold](http://localhost:5601/app/dashboards#/view/scipulse-monitoring) affiche les volumes contrôlés, les taux d'échec des attentes GX et la fraîcheur : derniers dépôts Bronze, derniers Parquet Silver, contrôles GX et dernier calcul Gold, avec leurs anciennetés.

Le taux est calculé sur les attentes, pas sur les lignes. Les rapports absents restent indisponibles. Une Silver en échec ne remplace pas le dernier Parquet valide.

Le monitoring est renouvelé après chaque cycle HN, planifié toutes les cinq minutes, et après le pipeline principal. Il reste un instantané ; les anciennetés sont évaluées à sa publication. Les données sont lues depuis MinIO et Elasticsearch.

Les 16 tests pipeline/monitoring passent dans Docker. Le rapport navigateur décrit les contrôles d'affichage. Les résultats sont dans `reports/monitoring_validation.json` et les mesures dans `reports/monitoring.json`. Voir [MONITORING.md](../docs/MONITORING.md). Les captures du monitoring sont enregistrées sous `monitoring-volumes-qualite.png` et `monitoring-fraicheur.png` dans `artifacts/screenshots/`. Elles présentent les volumes et taux GX, puis les dates et anciennetés.


In [ ]:
monitor = json.loads((ROOT / 'reports/monitoring.json').read_text(encoding='utf-8'))
monitor['published_at'], {s: d['quality'] for s, d in monitor['sources'].items()}

## Validation du pipeline

La [procédure de validation](../docs/VALIDATION.md) distingue les tests synthétiques, les contrôles Gold et les comparaisons analytiques. Aucun calcul Spark supplémentaire n’est nécessaire pour lire le rapport.

In [ ]:
final_validation = json.loads((ROOT / 'reports/final_validation.json').read_text(encoding='utf-8'))
final_validation['tests'], final_validation['analytical_queries']